# 🌍 Mframapa AI - Universal African PM2.5 Prediction Model

## Training Pipeline v2.1 (Diagnostic Mode)

**Changes in v2.1:**
- Simple train/val split (not GroupKFold) to test if data has signal
- Reduced regularization for better learning
- Correlation diagnostic before training

---

In [ ]:
# ============================================================
# CELL 0: MOUNT GOOGLE DRIVE & EXTRACT DATASET
# ============================================================

from google.colab import drive
import shutil
import zipfile
import os

drive.mount('/content/drive')

DRIVE_ZIP = '/content/drive/MyDrive/Mframapa/training_dataset.csv.zip'
LOCAL_DIR = '/content/data'
LOCAL_DATASET = f'{LOCAL_DIR}/training_dataset.csv'

os.makedirs(LOCAL_DIR, exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    print(f"[COPY] {DRIVE_ZIP} -> {LOCAL_DIR}/")
    local_zip = f'{LOCAL_DIR}/training_dataset.csv.zip'
    shutil.copy(DRIVE_ZIP, local_zip)
    print("[EXTRACT] Unzipping...")
    with zipfile.ZipFile(local_zip, 'r') as z:
        z.extractall(LOCAL_DIR)
    os.remove(local_zip)
    if os.path.exists(LOCAL_DATASET):
        size_gb = os.path.getsize(LOCAL_DATASET) / 1e9
        print(f"[SUCCESS] Dataset ready: {size_gb:.2f} GB")
    else:
        print(f"[WARN] Expected {LOCAL_DATASET} but not found.")
        print(os.listdir(LOCAL_DIR))
else:
    raise FileNotFoundError(f"Zip not found: {DRIVE_ZIP}")

In [ ]:
# ============================================================
# CELL 1: IMPORTS & CONFIGURATION
# ============================================================

import warnings; warnings.filterwarnings('ignore')
import gc, os
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"XGBoost: {xgb.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(f"GPU: {result.stdout.strip()}")

In [ ]:
# ============================================================
# CELL 2: DEFINE SCHEMA
# ============================================================

TARGET = 'pm25'

# Focus on features with actual correlation
SATELLITE_FEATURES = {
    'sat_wind_u': 'float32',    # Correlation: +0.37 (BEST)
    'sat_pblh': 'float32',      # Correlation: -0.24 (GOOD)
    'sat_no2': 'float32',       # Correlation: -0.16
    'sat_aot': 'float32',       # Correlation: +0.09
    'sat_wind_v': 'float32',    # Correlation: ~0
    'sat_humidity': 'float32',  # Correlation: ~0
}

GEO_FEATURES = {
    'lat': 'float32',
    'lon': 'float32',
    'pop_density': 'float32',
}

TEMPORAL_RAW = ['datetime']

ALL_COLUMNS = [TARGET] + list(SATELLITE_FEATURES.keys()) + list(GEO_FEATURES.keys()) + TEMPORAL_RAW
ALL_DTYPES = {**SATELLITE_FEATURES, **GEO_FEATURES, TARGET: 'float32'}

print(f"Schema: {len(ALL_COLUMNS)} columns")

In [ ]:
# ============================================================
# CELL 3: STREAMING DATA LOADER
# ============================================================

DATA_PATH = LOCAL_DATASET
CHUNK_SIZE = 50_000

header = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
print(f"[HEADER] {len(header)} columns in file")

usecols = [c for c in ALL_COLUMNS if c in header]
missing = set(ALL_COLUMNS) - set(usecols)
if missing:
    print(f"[WARN] Missing columns: {missing}")
print(f"[LOAD] {len(usecols)} columns: {usecols}")

print("[COUNT] Counting rows...")
with open(DATA_PATH, 'r') as f:
    total_rows = sum(1 for _ in f) - 1
print(f"[COUNT] {total_rows:,} rows")

chunks = []
rows_loaded = 0

print("[STREAM] Loading...")
for i, chunk in enumerate(pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    dtype=ALL_DTYPES,
    parse_dates=['datetime'] if 'datetime' in usecols else None,
    chunksize=CHUNK_SIZE,
    low_memory=True
)):
    chunks.append(chunk)
    rows_loaded += len(chunk)
    if (i + 1) % 20 == 0:
        gc.collect()
        pct = 100 * rows_loaded / total_rows
        print(f"  {rows_loaded:,} / {total_rows:,} ({pct:.1f}%)")

print("[CONCAT] Merging chunks...")
df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"[SUCCESS] Loaded {len(df):,} rows | Memory: {mem_gb:.2f} GB")

In [ ]:
# ============================================================
# CELL 4: CORRELATION DIAGNOSTIC (NEW)
# ============================================================

print("[DIAGNOSTIC] Checking feature correlations with PM2.5...")

# Sample for quick correlation check
sample = df.sample(min(100_000, len(df)), random_state=42)
numeric_cols = sample.select_dtypes(include=[np.number]).columns.tolist()

if TARGET in numeric_cols:
    correlations = sample[numeric_cols].corr()[TARGET].drop(TARGET).sort_values()
    print("\nCorrelations with PM2.5:")
    print(correlations)
    
    # Check if any feature has meaningful correlation
    max_corr = correlations.abs().max()
    if max_corr < 0.1:
        print("\n[WARN] No feature has correlation > 0.1. Model may fail.")
    elif max_corr < 0.3:
        print(f"\n[INFO] Best correlation: {max_corr:.2f}. Expect modest results.")
    else:
        print(f"\n[GOOD] Best correlation: {max_corr:.2f}. Signal detected!")

del sample
gc.collect()

In [ ]:
# ============================================================
# CELL 5: FEATURE ENGINEERING
# ============================================================

ENGINEERED_FEATURES = []

if 'datetime' in df.columns:
    print("[TIME] Engineering cyclical features...")
    
    month = df['datetime'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * month / 12).astype('float32')
    df['month_cos'] = np.cos(2 * np.pi * month / 12).astype('float32')
    ENGINEERED_FEATURES.extend(['month_sin', 'month_cos'])
    del month; gc.collect()
    
    if df['datetime'].dt.hour.max() > 0:
        hour = df['datetime'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * hour / 24).astype('float32')
        df['hour_cos'] = np.cos(2 * np.pi * hour / 24).astype('float32')
        ENGINEERED_FEATURES.extend(['hour_sin', 'hour_cos'])
        del hour; gc.collect()
    
    df.drop(columns=['datetime'], inplace=True)
    gc.collect()
    print(f"[TIME] Created: {ENGINEERED_FEATURES}")

FEATURES = []
for f in SATELLITE_FEATURES.keys():
    if f in df.columns:
        FEATURES.append(f)
for f in GEO_FEATURES.keys():
    if f in df.columns:
        FEATURES.append(f)
FEATURES.extend(ENGINEERED_FEATURES)

print(f"[FEATURES] {len(FEATURES)}: {FEATURES}")

In [ ]:
# ============================================================
# CELL 6: DATA CLEANING
# ============================================================

print(f"[CLEAN] Starting with {len(df):,} rows")

df[TARGET] = df[TARGET].replace([np.inf, -np.inf], np.nan)
gc.collect()

required_clean = [f for f in FEATURES if f in df.columns] + [TARGET]

bad_idx = df[df[required_clean].isna().any(axis=1)].index
print(f"[CLEAN] Dropping {len(bad_idx):,} rows with NaN...")
df.drop(bad_idx, inplace=True)
del bad_idx
gc.collect()

outlier_idx = df[(df[TARGET] <= 0) | (df[TARGET] >= 1000)].index
print(f"[CLEAN] Dropping {len(outlier_idx):,} PM2.5 outliers...")
df.drop(outlier_idx, inplace=True)
del outlier_idx
gc.collect()

df.reset_index(drop=True, inplace=True)
gc.collect()

print(f"[CLEAN] Final: {len(df):,} rows")

In [ ]:
# ============================================================
# CELL 7: ARRAY EXTRACTION
# ============================================================

print("[EXTRACT] Extracting arrays...")

y = df[TARGET].values.astype(np.float32)
df.drop(columns=[TARGET], inplace=True)
gc.collect()
print(f"  y: {y.shape} | range: [{y.min():.1f}, {y.max():.1f}] | mean: {y.mean():.1f}")

available_features = [f for f in FEATURES if f in df.columns]
X = df[available_features].values.astype(np.float32)
print(f"  X: {X.shape}")

del df
gc.collect()
print("[EXTRACT] DataFrame deleted")

In [ ]:
# ============================================================
# CELL 8: SIMPLE TRAIN/VAL SPLIT (NO GROUPS)
# ============================================================
# Using simple split to test if data has ANY predictive signal.
# ============================================================

print("[SPLIT] Simple 80/20 train/val split...")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

del X, y
gc.collect()

print(f"[SPLIT] Train: {X_train.shape} | Val: {X_val.shape}")
print(f"[SPLIT] Train mean PM2.5: {y_train.mean():.1f} | Val mean: {y_val.mean():.1f}")

In [ ]:
# ============================================================
# CELL 9: CREATE DMATRIX
# ============================================================

print("[DMATRIX] Creating...")

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=available_features)
del X_train, y_train; gc.collect()
print(f"  dtrain: {dtrain.num_row():,} x {dtrain.num_col()}")

dval = xgb.DMatrix(X_val, label=y_val, feature_names=available_features)
del X_val; gc.collect()
print(f"  dval: {dval.num_row():,} x {dval.num_col()}")

In [ ]:
# ============================================================
# CELL 10: MODEL HYPERPARAMETERS (REDUCED REGULARIZATION)
# ============================================================

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': RANDOM_STATE,
    
    # GPU
    'tree_method': 'hist',
    'device': 'cuda',
    'max_bin': 256,
    
    # Tree structure
    'max_depth': 8,               # Reduced from 10
    'max_leaves': 63,             # Reduced from 127
    'min_child_weight': 3,        # Reduced from 5
    'grow_policy': 'lossguide',
    
    # Learning
    'learning_rate': 0.05,        # Increased from 0.03
    
    # REDUCED Regularization
    'lambda': 1.0,                # Reduced from 5.0
    'alpha': 0.1,                 # Reduced from 0.5
    'gamma': 0.1,                 # Reduced from 1.0
    
    # Sampling
    'subsample': 0.8,             # Increased from 0.7
    'colsample_bytree': 0.8,      # Increased from 0.7
}

print("[PARAMS] Configured with REDUCED regularization")

In [ ]:
# ============================================================
# CELL 11: TRAIN MODEL
# ============================================================

print("="*60)
print("         MFRAMAPA AI - GPU TRAINING (v2.1)")
print("="*60)

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=3000,         # Reduced from 5000
    evals=[(dtrain, 'train'), (dval, 'valid')],
    early_stopping_rounds=100,    # Reduced from 150
    verbose_eval=25               # More frequent output
)

del dtrain; gc.collect()

print("="*60)
print(f"[DONE] Best Iteration: {model.best_iteration}")
print(f"[DONE] Best RMSE: {model.best_score:.4f}")
print("="*60)

In [ ]:
# ============================================================
# CELL 12: VALIDATION METRICS
# ============================================================

preds = model.predict(dval)

rmse = mean_squared_error(y_val, preds, squared=False)
mae = mean_absolute_error(y_val, preds)
r2 = r2_score(y_val, preds)

errors = np.abs(preds - y_val)
within_5 = 100 * np.mean(errors < 5)
within_10 = 100 * np.mean(errors < 10)
within_25 = 100 * np.mean(errors < 25)

# Baseline: just predict the mean
baseline_rmse = np.sqrt(np.mean((y_val - y_val.mean())**2))

print("\n" + "="*60)
print("         VALIDATION RESULTS")
print("="*60)
print(f"  RMSE:  {rmse:.4f} ug/m3 (Baseline: {baseline_rmse:.4f})")
print(f"  MAE:   {mae:.4f} ug/m3")
print(f"  R2:    {r2:.4f}")
print(f"")
print(f"  Within 5 ug/m3:  {within_5:.1f}%")
print(f"  Within 10 ug/m3: {within_10:.1f}%")
print(f"  Within 25 ug/m3: {within_25:.1f}%")
print("="*60)

if r2 > 0:
    print("\n[SUCCESS] Model beats baseline! R2 > 0")
else:
    print("\n[FAIL] Model is worse than baseline. Data may have no signal.")

del dval, preds; gc.collect()

In [ ]:
# ============================================================
# CELL 13: FEATURE IMPORTANCE
# ============================================================

importance = model.get_score(importance_type='gain')
importance_df = pd.DataFrame({
    'feature': list(importance.keys()),
    'gain': list(importance.values())
}).sort_values('gain', ascending=False)

importance_df['pct'] = 100 * importance_df['gain'] / importance_df['gain'].sum()

print("\n[IMPORTANCE] Feature Importance (Gain):")
print("-"*40)
for _, row in importance_df.iterrows():
    bar = '█' * int(row['pct'] / 2)
    print(f"  {row['feature']:15s} {row['pct']:5.1f}% {bar}")
print("-"*40)

In [ ]:
# ============================================================
# CELL 14: SAVE MODEL
# ============================================================

MODEL_NAME = 'universal_african_model.json'
LOCAL_MODEL = f'/content/{MODEL_NAME}'
DRIVE_MODEL = f'/content/drive/MyDrive/Mframapa/{MODEL_NAME}'

model.save_model(LOCAL_MODEL)
print(f"[SAVE] Local: {LOCAL_MODEL}")

shutil.copy(LOCAL_MODEL, DRIVE_MODEL)
print(f"[SAVE] Drive: {DRIVE_MODEL}")

print("\n" + "="*60)
print("         MFRAMAPA AI - TRAINING COMPLETE")
print("="*60)
print(f"  Model: {MODEL_NAME}")
print(f"  Trees: {model.best_iteration}")
print(f"  RMSE:  {rmse:.4f} ug/m3")
print(f"  R2:    {r2:.4f}")
print("="*60)